# Clean coded studies and get high level information


In [ ]:
import pandas as pd

## Ensure that all walkability factors are named consistently

In [ ]:
df = pd.read_csv('coded_studies.csv')

# clean factors
df['Factor'] = df['Factor'].str.title()

factor_map = {
    'Walking Amentities': 'Walking Amenities',
    'Cul-De-Sac': 'Cul-De-Sacs',
    'Intersection Density': 'Street Connectivity',
    'Recreation Density': 'Recreation Availability'
}

df['Factor'] = df['Factor'].replace(factor_map)

# remove factors we are not using: moveability index, pedestrian friendliness, retail floor area ratio, safety, PA environment
factor_studies = df[['Short Name', 'Factor']].drop_duplicates()
factor_studies = factor_studies['Factor'].value_counts()

included_factors = factor_studies[:-5]

df = df[df['Factor'].isin(included_factors.index)].copy()

### Get the country and continent for each study

In [ ]:
basic_info = pd.read_csv('basic_study_info.csv', index_col=0)
included = basic_info[basic_info['Include?'] == 'yes'].copy()

countries = included['Country'].str.split(';', expand=True)

# get the counts of each country
country_counts = countries[0].value_counts().to_dict()
country_counts_1 = countries[1].value_counts().to_dict()
country_counts_2 = countries[2].value_counts().to_dict()

# add the columns into a single dict
for country, count in country_counts_1.items():
    c = country.strip()
    if c in country_counts:
        country_counts[c] = country_counts[c] + count
    else:
        country_counts[c] = count

for country, count in country_counts_2.items():
    c = country.strip()
    if c in country_counts:
        country_counts[c] = country_counts[c] + count
    else:
        country_counts[c] = count

country_counts

{'USA': 20,
 'Canada': 9,
 'Australia': 5,
 'Belgium': 5,
 'Spain': 4,
 'New Zealand': 3,
 'Malaysia': 2,
 'Germany': 2,
 'Brazil': 1,
 'The Netherlands': 1,
 'Mexico': 1,
 'China': 1,
 'Uganda': 1,
 'South Africa': 1,
 'France': 1,
 'Italy': 1,
 'Sweden': 1}

Map countries to continents

In [ ]:
continent_map = {
    'USA': 'North America',
    'Canada': 'North America',
    'Germany': 'Europe',
    'Australia': 'Australia',
    'Belgium': 'Europe',
    'New Zealand': 'New Zealand',
    'Spain': 'Europe',
    'Malaysia': 'Asia',
    'Spain; France': 'Europe',
    'Germany; Italy; Sweden': 'Europe',
    'Brazil': 'South America',
    'The Netherlands': 'Europe',
    'Mexico': 'North America',
    'China': 'Asia',
    'Uganda': 'Africa',
    'South Africa': 'Africa'
}

included['Continent'] = included['Country'].map(continent_map)

included['Continent'].value_counts()

Continent
North America    30
Europe           12
Australia         5
Asia              3
New Zealand       3
Africa            2
South America     1
Name: count, dtype: int64

Combine continent with coded studies

In [ ]:
df = df.merge(included[['Short Name', 'Continent']], on='Short Name')

### Separate studies into below 12 and above 12

In [ ]:
def get_min(s):
    if '-' in s: 
        return s.split('-')[0]
    elif '(' in s: 
        split_list = s.split('(')
        mean = split_list[0]
        sd = split_list[1].split(')')[0]
        return pd.to_numeric(mean) - pd.to_numeric(sd)
    else: 
        return s

def get_max(s):
    if '-' in s: 
        return s.split('-')[1]
    elif '(' in s: 
        split_list = s.split('(')
        mean = split_list[0]
        sd = split_list[1].split(')')[0]
        return pd.to_numeric(mean) + pd.to_numeric(sd)
    else: 
        return s
    

df['min_age'] = pd.to_numeric(df['Ages'].apply(get_min))
df['max_age'] = pd.to_numeric(df['Ages'].apply(get_max))

# separate into below 12 and above 12
cutoff = 12
below = ((df['min_age'] < cutoff) & (df['max_age'] < cutoff))
above = ((df['min_age'] >= cutoff) & (df['max_age'] >= cutoff))

df['below_12'] = below
df['above_12'] = above

# drop the min and max age columns
df = df.drop(['min_age', 'max_age'], axis=1)

In [ ]:
age_tests = df[['Short Name', 'below_12', 'above_12']].drop_duplicates()

below_12 = age_tests.loc[age_tests['below_12'], ['Short Name', 'below_12']]
above_12 = age_tests.loc[age_tests['above_12'], ['Short Name', 'above_12']]
neither = age_tests[~age_tests['above_12'] & ~age_tests['below_12']]

print('Below 12: ', len(below_12))
print('Above 12: ', len(above_12))
print('Neither: ', len(neither))


Below 12:  17
Above 12:  26
Neither:  20


### Walkability Category

Clean the factors

In [ ]:
factor_prevalence = df.groupby('Factor')[['Tested']].sum()

factor_prevalence.sort_values(by='Tested', ascending=False)

,Tested
Factor,
Walkability Index,331
Recreation Availability,288
Traffic Safety,247
Walking Amenities,166
Aesthetics,164
Poi Availability,163
Street Connectivity,114
Residential Density,94
Crime Safety,84


In [ ]:
factors = factor_prevalence.merge(factor_studies, left_index=True, right_index=True)
factors = factors.rename(columns = {'count': 'Number of studies', 'Tested': 'Number of associations'})

factors = factors.sort_values('Number of studies', ascending=False)

factors.to_csv('output/factor_associations.csv')

factors

,Number of associations,Number of studies
Factor,,
Walkability Index,331,38
Recreation Availability,288,31
Traffic Safety,247,25
Aesthetics,164,23
Poi Availability,163,22
Walking Amenities,166,22
Residential Density,94,22
Street Connectivity,114,21
Crime Safety,84,16


Total tested

In [ ]:
print(factors[['Number of associations']].sum())

print(factors[['Number of associations']].sum() / 56)

Number of associations    1827
dtype: int64
Number of associations    32.625
dtype: float64


### PA Measurement Category

In [ ]:
pa_meas_map = {
    'accelerometer': 'objective',
    'accelerometer/pedometer': 'objective',
    'pedometer': 'objective',
    'accelerometer; parent-report': 'objective',
    '?': 'parent and self report'
}

df['PA Measurement Category'] = df['PA Measurement'].replace(pa_meas_map)
df['PA Measurement Category'].value_counts()

PA Measurement Category
objective                 290
self-report               236
parent-report             185
parent and self report      5
Name: count, dtype: int64

### Walkability Measurement Category

In [ ]:
df['Factor Measurement'] = df['Factor Measurement'].str.capitalize()
df['Factor Measurement'].value_counts()

Factor Measurement
Parent-report    255
Gis              200
Audit            139
Self-report      115
Gis; audit         6
Imagery            1
Name: count, dtype: int64

## Save cleaned dataframe

In [ ]:
df.to_csv('output/cleaned_data.csv')